# Week 8 — Day 5: The Price is Right (Final UI)

Wire everything from days 1-4 into a polished Gradio UI built with `gr.Blocks`. We build it up incrementally — three iterations — then hand off to the standalone `price_is_right.py` for the production version.

## Imports

In [ ]:
import gradio as gr
from deal_agent_framework import DealAgentFramework
from agents.deals import Opportunity, Deal

## Iteration 1 — title + subtitle only

In [ ]:
with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    with gr.Row():
        gr.Markdown(
            '<div style="text-align: center;font-size:24px">The Price is Right - Deal Hunting Agentic AI</div>'
        )
    with gr.Row():
        gr.Markdown(
            '<div style="text-align: center;font-size:14px">'
            'Autonomous agent framework that finds online deals, collaborating with a proprietary fine-tuned LLM '
            'deployed on Modal, and a RAG pipeline with a frontier model and Chroma.</div>'
        )

ui.launch(inbrowser=True)

## Iteration 2 — add a deals dataframe seeded with one example opportunity

In [ ]:
with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [
            [opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url]
            for opp in opps
        ]

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">"The Price is Right" - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])

ui.launch(inbrowser=True)

## Iteration 3 — wire row-click to send a push notification

Selecting a row triggers `agent_framework.planner.messenger.alert(opportunity)`.

In [ ]:
agent_framework = DealAgentFramework()
agent_framework.init_agents_as_needed()

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
    initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
    opportunities = gr.State([initial_opportunity])

    def get_table(opps):
        return [
            [opp.deal.product_description, opp.deal.price, opp.estimate, opp.discount, opp.deal.url]
            for opp in opps
        ]

    def do_select(opportunities, selected_index: gr.SelectData):
        row = selected_index.index[0]
        opportunity = opportunities[row]
        agent_framework.planner.messenger.alert(opportunity)

    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:24px">"The Price is Right" - Deal Hunting Agentic AI</div>')
    with gr.Row():
        gr.Markdown('<div style="text-align: center;font-size:14px">Deals surfaced so far:</div>')
    with gr.Row():
        opportunities_dataframe = gr.Dataframe(
            headers=["Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True,
            column_widths=[4, 1, 1, 1, 2],
            row_count=10,
            col_count=5,
            max_height=400,
        )

    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])

ui.launch(inbrowser=True)

## Run the full production app

`price_is_right.py` is the final standalone version with logging, vector-DB visualization, scheduled re-runs, and the full multi-pane layout.

In [ ]:
# Trim memory back to its 2 seed deals
from deal_agent_framework import DealAgentFramework
DealAgentFramework.reset_memory()

In [ ]:
import logging
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
!uv run price_is_right.py